# V2 - Custom CNN
## Plant Disease Detection — PyTorch CNN from scratch (38 classes)

In [ ]:
# ── Cell 1: Setup & Imports ──────────────────────────────────────────────────
import sys, os, json, time, copy
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix,
                             classification_report)

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path("..").resolve()
DATA_DIR     = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR  = PROJECT_ROOT / "results"
MODELS_DIR   = RESULTS_DIR / "models" / "v2_custom_cnn"
METRICS_DIR  = RESULTS_DIR / "metrics"
PLOTS_DIR    = RESULTS_DIR / "plots"

for d in [MODELS_DIR, METRICS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Device (MPS on Apple Silicon, CUDA if available, otherwise CPU) ───────────
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_SIZE    = 224
BATCH_SIZE  = 32   # MPS handles larger batches than CPU
LR          = 1e-3
EPOCHS      = 10
PATIENCE    = 2
NUM_WORKERS = 0    # on MPS the DataLoaders run with num_workers=0

FAST_MODE = True
print(f"Fast mode: {FAST_MODE}")


In [ ]:
# ── Cell 2: Dataset & DataLoaders ─────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=10, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset = datasets.ImageFolder(DATA_DIR / "train", transform=train_transform)
val_dataset   = datasets.ImageFolder(DATA_DIR / "val",   transform=val_test_transform)
test_dataset  = datasets.ImageFolder(DATA_DIR / "test",  transform=val_test_transform)

# optional: subsample for fast tests on CPU
if FAST_MODE:
    def subsample(ds, frac=0.1):
        n = max(1, int(len(ds) * frac))
        idx = torch.randperm(len(ds))[:n].tolist()
        return torch.utils.data.Subset(ds, idx)
    train_dataset = subsample(train_dataset)
    val_dataset   = subsample(val_dataset)
    test_dataset  = subsample(test_dataset)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

CLASS_NAMES = train_dataset.classes if hasattr(train_dataset, "classes") else train_dataset.dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

print(f"Classes: {NUM_CLASSES}")
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")


In [ ]:
# ── Cell 3: CNN Model (~0.66M parameters) ────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class PlantDiseaseNet(nn.Module):
    """4 Conv+BN+ReLU blocks → GlobalAvgPool → Dense 512→256→num_classes."""
    def __init__(self, num_classes=38):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3,   32), nn.MaxPool2d(2),   # → (B, 32, 112, 112)
            ConvBlock(32,  64), nn.MaxPool2d(2),   # → (B, 64,  56,  56)
            ConvBlock(64, 128), nn.MaxPool2d(2),   # → (B,128,  28,  28)
            ConvBlock(128,256),                     # → (B,256,  28,  28)
        )
        self.pool = nn.AdaptiveAvgPool2d(1)         # → (B,256,   1,   1)
        self.classifier = nn.Sequential(
            nn.Linear(256, 512), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(512, 256), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)


model = PlantDiseaseNet(num_classes=NUM_CLASSES).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}  (~{total_params/1e6:.1f}M)")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.1, patience=5)
print("Model, loss and optimizer ready ✅")


In [ ]:
# ── Cell 4: Training Loop with Early Stopping (with resume) ──────────────────
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total   += images.size(0)
    return running_loss / total, correct / total


def validate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total   += images.size(0)
    return running_loss / total, correct / total


checkpoint_path = MODELS_DIR / "checkpoint.pth"

best_val_loss  = float("inf")
start_epoch    = 1
history        = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
print("No checkpoint found, training from scratch.")
best_model_wts = copy.deepcopy(model.state_dict())
patience_count = 0
t0 = time.time()

print(f"Training on {DEVICE} — epoch {start_epoch}→{EPOCHS}, early stopping patience={PATIENCE}\n")

for epoch in range(start_epoch, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    vl_loss, vl_acc = validate(model, val_loader, criterion, DEVICE)
    scheduler.step(vl_loss)

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(vl_acc)

    improved = vl_loss < best_val_loss
    if improved:
        best_val_loss  = vl_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save({"epoch": epoch,
                    "model_state_dict": best_model_wts,
                    "val_loss": best_val_loss,
                    "val_acc": vl_acc,
                    "history": history}, checkpoint_path)
        patience_count = 0
        tag = " ✓ saved"
    else:
        patience_count += 1
        tag = f" (patience {patience_count}/{PATIENCE})"

    print(f"Epoch {epoch:3d}/{EPOCHS} | "
          f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
          f"val loss {vl_loss:.4f} acc {vl_acc:.4f}{tag}")

    if patience_count >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

elapsed = time.time() - t0
print(f"\nTraining complete in {elapsed/60:.1f} additional min")
print(f"Best val loss: {best_val_loss:.4f}")

with open(METRICS_DIR / "v2_fast_history.json", "w") as f:
    json.dump(history, f, indent=2)
print("History saved ✅")


In [ ]:
# ── Cell 5: Evaluate on the Test Set ──────────────────────────────────────────
# load the best checkpoint
ckpt = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        preds  = model(images).argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
rec  = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
f1   = f1_score(all_labels, all_preds, average="weighted", zero_division=0)

metrics = {
    "model":    "V2_CustomCNN",
    "accuracy": round(acc,  4),
    "precision": round(prec, 4),
    "recall":   round(rec,  4),
    "f1":       round(f1,   4),
    "best_epoch": int(ckpt["epoch"]),
    "training_time_min": round(elapsed / 60, 2),
}
with open(METRICS_DIR / "v2_fast_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("── V2 Metrics ─────────────────────────")
for k, v in metrics.items():
    print(f"  {k}: {v}")
print("Saved to results/metrics/v2_fast_metrics.json ✅")


In [ ]:
# ── Cell 6: Training Curves ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_ran = range(1, len(history["train_loss"]) + 1)

axes[0].plot(epochs_ran, history["train_loss"], label="Train Loss")
axes[0].plot(epochs_ran, history["val_loss"],   label="Val Loss")
axes[0].set_title("Loss per Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_ran, history["train_acc"], label="Train Accuracy")
axes[1].plot(epochs_ran, history["val_acc"],   label="Val Accuracy")
axes[1].set_title("Accuracy per Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("V2 Custom CNN — Training Curves", fontsize=14, fontweight="bold")
plt.tight_layout()
out_path = PLOTS_DIR / "v2_fast_training_curves.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path} ✅")


In [ ]:
# ── Cell 7: Confusion Matrix ──────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(20, 18))
sns.heatmap(cm, annot=False, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("True", fontsize=12)
ax.set_title(f"V2 Custom CNN — Confusion Matrix\nAccuracy: {acc:.4f}", fontsize=14)
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0,  fontsize=7)
plt.tight_layout()
out_path = PLOTS_DIR / "v2_fast_confusion_matrix.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path} ✅")

# Per-class F1
f1_per_class = f1_score(all_labels, all_preds, average=None, zero_division=0)
fig2, ax2 = plt.subplots(figsize=(18, 5))
bars = ax2.bar(CLASS_NAMES, f1_per_class, color="steelblue")
ax2.set_xticks(range(len(CLASS_NAMES)))
ax2.set_xticklabels(CLASS_NAMES, rotation=90, fontsize=7)
ax2.set_ylabel("F1 Score")
ax2.set_title("V2 Custom CNN — Per-class F1")
ax2.set_ylim(0, 1.05)
ax2.axhline(f1_per_class.mean(), color="red", linestyle="--", label=f"Mean: {f1_per_class.mean():.3f}")
ax2.legend()
plt.tight_layout()
out_path2 = PLOTS_DIR / "v2_fast_f1_per_class.png"
plt.savefig(out_path2, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path2} ✅")
print("\nDone! Notebook 02 complete.")
